# Notebook 05: Testing and evaluation (internal test, Ghanaian external test E3, abstention E4)

**Framework component:** F6, evaluation and reporting. Needs `frozen_choices_v2.json` from Notebook 04.

This is the **first** notebook that loads the internal test partition and the Ghanaian (D3) images. Nothing chosen here can change a model, format, temperature or threshold: they are read from the frozen file, whose SHA-256 is checked first, and each model file is checked against the hash recorded when it was frozen.

It reports, as defined in docs/METHODS.md:

1. accuracy, balanced accuracy, macro-F1, per-class precision, recall and F1, confusion matrices, NLL, Brier score and ECE (15 bins), for all 18 files, internal and external, with seed mean and standard deviation;
2. abstention at the frozen threshold: coverage, selective accuracy and risk, correct accepted yield, coverage per class;
3. for the seed-42 models, 95% group-bootstrap intervals (2,000 resamples) and paired FP32 vs FP16 and FP32 vs INT8 differences;
4. the two maize source analyses: healthy-maize recall for D4 and D6 separately, and a simple classifier that predicts the source of healthy images from frozen features;
5. Figures 5.1 and 5.2 and a stratified sheet of correct, wrong and rejected predictions for inspection.

On Ghanaian maize, which has only streak virus and healthy images, the models keep all five outputs: predicting rust, blight or leaf spot counts as an error. Absent classes are reported as **not evaluated**.

**Runtime:** A100 or L4 (CPU cores matter most). About 30 to 60 minutes. Finished scoring is skipped if you run all again.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, zipfile
import numpy as np, pandas as pd
import tensorflow as tf, keras
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'; PROTOCOL_VERSION = 'v2'
OUT = f'{DRIVE}/records/nb05'; LOG = f'{OUT}/logits'; MODELS = f'{DRIVE}/models'; RUNS = f'{DRIVE}/runs'
for d in [OUT, LOG, WORK]: os.makedirs(d, exist_ok=True)
try: gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception: gpu_name = 'none'
try:
    from ai_edge_litert.interpreter import Interpreter, OpResolverType; litert_pkg = 'ai_edge_litert'
except Exception:
    Interpreter = tf.lite.Interpreter; OpResolverType = tf.lite.experimental.OpResolverType; litert_pkg = 'tf.lite (bundled with TensorFlow)'
env = {'notebook': '05_testing_evaluation', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'litert_interpreter': litert_pkg,
       'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
try:
    import sklearn; env['scikit_learn'] = sklearn.__version__
except Exception: pass
json.dump(env, open(f'{OUT}/environment_nb05.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Helpers: calibration (Notebook 04) and evaluation metrics, both tested before delivery
# ---- OCDF F4 calibration, abstention and selection helpers (Notebook 04) ----
import numpy as np

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T=1.0):
    p = softmax(z, T)
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))

def fit_temperature(z, y, lo=0.05, hi=20.0):
    """Calibration: positive T minimising negative log-likelihood on the calibration partition (Guo et al., 2017).
    Golden-section search on log T (NLL is unimodal in T for fixed logits)."""
    a, b = np.log(lo), np.log(hi); g = (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll(z, y, np.exp(c)), nll(z, y, np.exp(d))
    for _ in range(80):
        if fc < fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = nll(z, y, np.exp(c))
        else: a, c, fc = c, d, fd; d = a + g * (b - a); fd = nll(z, y, np.exp(d))
    T = float(np.exp((a + b) / 2))
    return {'T': round(T, 4), 'nll_before': round(nll(z, y, 1.0), 4), 'nll_after': round(nll(z, y, T), 4)}

def brier(p, y):
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def ece(p, y, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[0]) & (conf <= edges[1])
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def threshold_table(p, y, groups, thresholds=None):
    """Coverage, selective accuracy and accepted groups for each threshold (see docs/METHODS.md)."""
    thresholds = np.round(np.arange(0.50, 0.995, 0.01), 2) if thresholds is None else thresholds
    conf = p.max(1); pred = p.argmax(1); groups = np.asarray(groups); rows = []
    for t in thresholds:
        acc_mask = conf >= t; n_acc = int(acc_mask.sum())
        rows.append({'threshold': float(t), 'coverage': n_acc / len(y),
                     'selective_accuracy': float((pred[acc_mask] == y[acc_mask]).mean()) if n_acc else None,
                     'selective_risk': float((pred[acc_mask] != y[acc_mask]).mean()) if n_acc else None,
                     'correct_accepted_yield': float(((pred == y) & acc_mask).mean()),
                     'accepted_groups': int(len(set(groups[acc_mask]))), 'accepted': n_acc})
    return rows

def choose_threshold(rows, min_acc=0.95, min_cov=0.50, min_groups=50, fallback=0.90):
    """Highest-coverage threshold with selective accuracy >= 0.95, coverage >= 0.50 and >= 50 accepted groups;
    otherwise 'target infeasible' and the predeclared 0.90 for description only."""
    ok = [r for r in rows if r['selective_accuracy'] is not None and r['selective_accuracy'] >= min_acc
          and r['coverage'] >= min_cov and r['accepted_groups'] >= min_groups]
    if ok:
        best = max(ok, key=lambda r: (r['coverage'], -r['threshold']))
        return {'threshold': best['threshold'], 'status': 'target met', **{k: best[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}
    fb = min(rows, key=lambda r: abs(r['threshold'] - fallback))
    return {'threshold': fallback, 'status': 'target infeasible (0.90 used for description only)',
            **{k: fb[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}

def per_class_coverage(p, y, t, classes):
    conf = p.max(1); out = {}
    for i, c in enumerate(classes):
        m = y == i
        out[c] = round(float((conf[m] >= t).mean()), 4) if m.any() else None
    return out

def select_architecture(agg_rows, size_bytes=None, margin=0.01):
    """Pre-declared selection rule: highest mean selection macro-F1; within `margin` of the best, the smaller converted file."""
    best = max(agg_rows, key=lambda r: r['mean_macro_f1'])
    close = [r for r in agg_rows if best['mean_macro_f1'] - r['mean_macro_f1'] <= margin + 1e-12]
    if len(close) > 1 and size_bytes:
        pick = min(close, key=lambda r: size_bytes[r['architecture']])
        return pick['architecture'], f'within {margin} of best; smallest FP32 file chosen'
    return best['architecture'], (f"highest mean selection macro-F1 ({best['mean_macro_f1']:.4f}); "
                                   f"no other architecture within {margin}")

# ---- OCDF F6 evaluation helpers (Notebook 05): metrics, abstention, group bootstrap ----
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def full_metrics(p, y, eval_labels, n_out, threshold=None, groups=None):
    """Evaluation metrics (docs/METHODS.md). `eval_labels`: class indices present in this evaluation set (others are 'not evaluated').
    Predictions keep all n_out outputs (never masked), so predicting an absent class counts as an error."""
    pred = p.argmax(1); L = list(eval_labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    out = {'n_images': int(len(y)), 'n_groups': int(len(set(groups))) if groups is not None else None,
           'accuracy': float(accuracy_score(y, pred)),
           'balanced_accuracy': float(np.mean([np.mean(pred[y == c] == c) for c in L])),
           'macro_f1': float(f1_score(y, pred, labels=L, average='macro', zero_division=0)),
           'per_class': {int(c): {'precision': float(a), 'recall': float(b), 'f1': float(d), 'support': int(s)}
                         for c, a, b, d, s in zip(L, pr, rc, f1, sup)},
           'confusion': confusion_matrix(y, pred, labels=list(range(n_out))).tolist(),
           'nll': float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))),
           'brier': float(np.mean(np.sum((p - np.eye(n_out)[y]) ** 2, axis=1))),
           'ece_15': ece15(p, y)}
    if threshold is not None:
        acc = p.max(1) >= threshold; n = int(acc.sum())
        out.update({'threshold': float(threshold), 'coverage': float(acc.mean()),
                    'selective_accuracy': float((pred[acc] == y[acc]).mean()) if n else None,
                    'selective_risk': float((pred[acc] != y[acc]).mean()) if n else None,
                    'correct_accepted_yield': float(((pred == y) & acc).mean()),
                    'coverage_per_class': {int(c): float(acc[y == c].mean()) for c in L}})
    return out

def ece15(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float); e = 0.0
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fast_accuracy(y, pred):
    return float(np.mean(y == pred))

def fast_macro_f1(y, pred, labels, n_out):
    """Macro-F1 over `labels` from a bincount confusion matrix; identical to sklearn f1_score(average='macro', zero_division=0)."""
    cm = np.bincount(y * n_out + pred, minlength=n_out * n_out).reshape(n_out, n_out)
    tp = np.diag(cm)[labels].astype(float); fp = cm[:, labels].sum(0) - tp; fn = cm[labels, :].sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)))

def group_bootstrap(stat_fn, groups, n_boot=2000, seed=2026):
    """95% percentile interval of stat_fn(index_array), resampling whole groups with replacement (Efron & Tibshirani, 1994)."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind='stable'); lengths = np.bincount(inv); starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq)); lens = lengths[pick]
        offs = np.repeat(starts[pick] - np.cumsum(lens) + lens, lens) + np.arange(lens.sum())
        vals[b] = stat_fn(order[offs])
    return {'estimate': float(stat_fn(np.arange(len(groups)))), 'ci95': [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))],
            'n_boot': n_boot, 'n_groups': int(len(uniq))}

def reliability_bins(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    idx = np.minimum((conf * bins).astype(int), bins - 1); rows = []
    for b in range(bins):
        m = idx == b
        rows.append({'bin_low': b / bins, 'bin_high': (b + 1) / bins, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()) if m.any() else None, 'accuracy': float(correct[m].mean()) if m.any() else None})
    return rows

def risk_coverage(p, y):
    order = np.argsort(-p.max(1)); correct = (p.argmax(1) == y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k

In [ ]:
# 3. Integrity: frozen choices, manifest and every model file must match their recorded SHA-256
fpath = f'{DRIVE}/records/frozen_choices_{PROTOCOL_VERSION}.json'
fsha = hashlib.sha256(open(fpath, 'rb').read()).hexdigest()
assert fsha == open(f'{DRIVE}/records/frozen_choices_{PROTOCOL_VERSION}.sha256').read().strip(), 'frozen choices file changed'
F = json.load(open(fpath)); protocol = json.load(open(f'{DRIVE}/records/protocol_{PROTOCOL_VERSION}.json'))
mpath = f'{DRIVE}/manifests/{protocol["manifest"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == protocol['manifest_sha256'] == F['manifest_sha256']
V = pd.DataFrame(F['all_variants'])
for r in V.itertuples():
    assert hashlib.sha256(open(f'{DRIVE}/{r.file}', 'rb').read()).hexdigest() == r.sha256, f'{r.file} changed after freezing'
print('frozen choices', fsha[:16], '| manifest and all', len(V), 'model files match their frozen hashes')
print('deployed instance:', {c: (d['architecture'], d['seed'], d['format'], d['temperature_T'], d['threshold']) for c, d in F['deployed_instance'].items()})
M = pd.read_csv(mpath, dtype=str)
for src in ['D1', 'D3', 'D4', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}
IMG = 224; SEEDS = [17, 42, 2026]; FORMATS = ['fp32', 'fp16', 'int8']
SETS = {}
for crop in CLASSES:
    SETS[(crop, 'internal')] = M[(M.crop == crop) & (M.partition == 'test')].sort_values('sample_id').reset_index(drop=True)
    SETS[(crop, 'external')] = M[(M.crop == crop) & (M.partition == 'external')].sort_values('sample_id').reset_index(drop=True)
for k, d in SETS.items(): print(k, len(d), 'images,', d.group_id.nunique(), 'groups,', d.research_label.value_counts().to_dict())

In [ ]:
# 4. Score every frozen file on the internal test and on D3 (logits saved; resumable)
def load_img(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG, IMG, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)
def load_array(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(load_img, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
def _run(b, X, threads, batch, xnnpack):
    it = Interpreter(model_content=b, num_threads=threads) if xnnpack else \
         Interpreter(model_content=b, num_threads=threads, experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    it.allocate_tensors(); i, o = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
    it.resize_tensor_input(i, [batch, IMG, IMG, 3]); it.allocate_tensors(); out = []
    for s in range(0, len(X), batch):
        xb = X[s:s + batch].astype(np.float32); n = len(xb)
        if n < batch: xb = np.concatenate([xb, np.zeros((batch - n, IMG, IMG, 3), np.float32)])
        it.set_tensor(i, xb); it.invoke(); out.append(it.get_tensor(o)[:n].copy())
    return np.concatenate(out)
def run_tflite(b, X, batch=32):
    try: return _run(b, X, os.cpu_count(), batch, True)
    except Exception: return _run(b, X, os.cpu_count(), batch, False)

for (crop, setname), df in SETS.items():
    X = None
    for r in V[V.crop == crop].itertuples():
        out = f'{LOG}/{crop}_s{r.seed}_{r.format}_{setname}.npy'
        if os.path.exists(out): continue
        if X is None: X = load_array(df)
        t0 = time.time(); z = run_tflite(open(f'{DRIVE}/{r.file}', 'rb').read(), X); np.save(out, z)
        print(f'{crop} {setname} s{r.seed} {r.format}: {len(z)} images, {time.time()-t0:.0f}s')
    del X; gc.collect()

In [ ]:
# 5. Metrics for all 18 files (frozen T and threshold of each file), and seed mean / SD per format
ROWS = []; DETAIL = {}
for (crop, setname), df in SETS.items():
    cl = CLASSES[crop]; y = np.array([cl.index(l) for l in df.research_label]); present = sorted(set(y.tolist()))
    for r in V[V.crop == crop].itertuples():
        z = np.load(f'{LOG}/{crop}_s{r.seed}_{r.format}_{setname}.npy'); p = softmax(z, r.T)
        m = full_metrics(p, y, present, len(cl), threshold=r.threshold, groups=df.group_id.values)
        m['not_evaluated'] = [cl[i] for i in range(len(cl)) if i not in present]
        DETAIL[f'{crop}_{setname}_s{r.seed}_{r.format}'] = m
        ROWS.append({'crop': crop, 'set': setname, 'seed': r.seed, 'format': r.format, 'n_images': m['n_images'], 'n_groups': m['n_groups'],
                     'accuracy': m['accuracy'], 'balanced_accuracy': m['balanced_accuracy'], 'macro_f1': m['macro_f1'],
                     'nll': m['nll'], 'brier': m['brier'], 'ece_15': m['ece_15'], 'T': r.T, 'threshold': r.threshold,
                     'coverage': m['coverage'], 'selective_accuracy': m['selective_accuracy'], 'correct_accepted_yield': m['correct_accepted_yield'],
                     **{f'f1_{cl[c]}': v['f1'] for c, v in m['per_class'].items()},
                     **{f'recall_{cl[c]}': v['recall'] for c, v in m['per_class'].items()},
                     **{f'coverage_{cl[c]}': v for c, v in m['coverage_per_class'].items()}})
T5 = pd.DataFrame(ROWS); T5.to_csv(f'{OUT}/results_all_variants.csv', index=False)
SEEDSUM = T5.groupby(['crop', 'set', 'format'])[['accuracy', 'balanced_accuracy', 'macro_f1', 'ece_15', 'coverage', 'selective_accuracy']].agg(['mean', 'std']).round(4)
SEEDSUM.to_csv(f'{OUT}/results_seed_mean_sd.csv'); print(SEEDSUM)
json.dump(DETAIL, open(f'{OUT}/results_detail.json', 'w'), indent=1)

In [ ]:
# 6. Seed-42 models: 95% group-bootstrap intervals (2,000 resamples) and paired format differences
from sklearn.metrics import f1_score as _f1, accuracy_score as _acc
B = []
for (crop, setname), df in SETS.items():
    cl = CLASSES[crop]; y = np.array([cl.index(l) for l in df.research_label]); present = sorted(set(y.tolist())); g = df.group_id.values
    P = {}
    for fmt in FORMATS:
        r = V[(V.crop == crop) & (V.seed == 42) & (V.format == fmt)].iloc[0]
        P[fmt] = (softmax(np.load(f'{LOG}/{crop}_s42_{fmt}_{setname}.npy'), r['T']), r.threshold)
    PRED = {f: P[f][0].argmax(1) for f in FORMATS}; K = len(cl)
    def acc(f): return lambda i: fast_accuracy(y[i], PRED[f][i])
    def mf1(f): return lambda i: fast_macro_f1(y[i], PRED[f][i], present, K)
    def cov(f): return lambda i: float((P[f][0][i].max(1) >= P[f][1]).mean())
    def sacc(f):
        def s(i):
            a = P[f][0][i].max(1) >= P[f][1]
            return float((P[f][0][i][a].argmax(1) == y[i][a]).mean()) if a.any() else np.nan
        return s
    for fmt in FORMATS:
        for name, fn in [('accuracy', acc(fmt)), ('macro_f1', mf1(fmt)), ('coverage', cov(fmt)), ('selective_accuracy', sacc(fmt))]:
            B.append({'crop': crop, 'set': setname, 'comparison': fmt, 'metric': name, **group_bootstrap(fn, g)})
    for other in ['fp16', 'int8']:
        for name, a, b in [('accuracy', acc('fp32'), acc(other)), ('macro_f1', mf1('fp32'), mf1(other))]:
            B.append({'crop': crop, 'set': setname, 'comparison': f'fp32 minus {other} (paired)', 'metric': name,
                      **group_bootstrap(lambda i, a=a, b=b: a(i) - b(i), g)})
    print(crop, setname, 'done')
BT = pd.DataFrame(B); BT['ci_low'] = BT.ci95.str[0]; BT['ci_high'] = BT.ci95.str[1]
BT.drop(columns=['ci95']).to_csv(f'{OUT}/bootstrap_seed42.csv', index=False)
print(BT[['crop', 'set', 'comparison', 'metric', 'estimate', 'ci_low', 'ci_high', 'n_groups']].round(4).to_string(index=False))

In [ ]:
# 7. Maize source analyses 
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
SRC = {}
# (a) recall of healthy maize separately for D4 and D6, internal test, every file
df = SETS[('maize', 'internal')]; cl = CLASSES['maize']; y = np.array([cl.index(l) for l in df.research_label])
h = (df.research_label == 'M05').values; rows = []
for r in V[V.crop == 'maize'].itertuples():
    pred = softmax(np.load(f'{LOG}/maize_s{r.seed}_{r.format}_internal.npy'), r.T).argmax(1)
    for src in ['D4', 'D6']:
        m = h & (df.source_id == src).values
        rows.append({'seed': r.seed, 'format': r.format, 'source': src, 'healthy_images': int(m.sum()), 'healthy_recall': float((pred[m] == 4).mean())})
SRC['healthy_recall_by_source'] = rows
print(pd.DataFrame(rows).pivot_table(index=['seed', 'format'], columns='source', values='healthy_recall').round(4))
# (b) can frozen features of the selected maize model (seed 42) tell D4 healthy from D6 healthy?
dep = F['deployed_instance']['maize']
m = keras.models.load_model(f'{RUNS}/maize_{dep["architecture"]}_s{dep["seed"]}/best_model.keras')
feat = keras.Model(m.inputs, m.get_layer('gap').output)
def feats(d): return feat.predict(load_array(d).astype(np.float32), batch_size=64, verbose=0)
tr = M[(M.crop == 'maize') & (M.partition == 'train') & (M.research_label == 'M05')]
te = M[(M.crop == 'maize') & (M.partition == 'test') & (M.research_label == 'M05')]
Xtr, Xte = feats(tr), feats(te); ytr, yte = (tr.source_id == 'D6').values.astype(int), (te.source_id == 'D6').values.astype(int)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced')).fit(Xtr, ytr)
pp = clf.predict_proba(Xte)[:, 1]
SRC['source_classifier'] = {'features': f'{dep["architecture"]} seed {dep["seed"]} global-average-pooled features (Keras model, frozen)',
                            'train_images': {'D4': int((ytr == 0).sum()), 'D6': int((ytr == 1).sum())},
                            'test_images': {'D4': int((yte == 0).sum()), 'D6': int((yte == 1).sum())},
                            'test_accuracy': float(((pp >= 0.5) == yte).mean()), 'test_balanced_accuracy': float(balanced_accuracy_score(yte, pp >= 0.5)),
                            'test_auc': float(roc_auc_score(yte, pp))}
print(json.dumps(SRC['source_classifier'], indent=1))
json.dump(SRC, open(f'{OUT}/maize_source_analysis.json', 'w'), indent=1)
del m, feat; gc.collect()

In [ ]:
# 8. Figures 5.1 (confusion matrices) and 5.2 (reliability and risk-coverage), and the inspection sheet
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
DEP = F['deployed_instance']
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (crop, setname) in zip(axes, [('cassava', 'internal'), ('cassava', 'external'), ('maize', 'internal'), ('maize', 'external')]):
    d = DEP[crop]; cl = CLASSES[crop]; det = DETAIL[f'{crop}_{setname}_s{d["seed"]}_{d["format"]}']
    cm = np.array(det['confusion'], dtype=float); rows = cm.sum(1, keepdims=True); norm = np.divide(cm, rows, out=np.zeros_like(cm), where=rows > 0)
    ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(cl)):
        for j in range(len(cl)):
            if rows[i, 0] > 0: ax.text(j, i, f'{int(cm[i, j])}\n{norm[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if norm[i, j] > .5 else 'black')
    ax.set_xticks(range(len(cl))); ax.set_xticklabels(cl); ax.set_yticks(range(len(cl)))
    ax.set_yticklabels([c if rows[i, 0] > 0 else f'{c} (n/a)' for i, c in enumerate(cl)])
    ax.set_title(f'{crop.capitalize()}, {"internal test" if setname == "internal" else "Ghana (D3)"}\n{d["architecture"]} s{d["seed"]} {d["format"].upper()}, n={det["n_images"]}', fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.savefig(f'{OUT}/figure_5_1_confusion.png', dpi=200); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for crop, mk in [('cassava', 'o'), ('maize', 's')]:
    d = DEP[crop]; cl = CLASSES[crop]
    for setname, ls in [('internal', '-'), ('external', '--')]:
        df = SETS[(crop, setname)]; y = np.array([cl.index(l) for l in df.research_label])
        p = softmax(np.load(f'{LOG}/{crop}_s{d["seed"]}_{d["format"]}_{setname}.npy'), d['temperature_T'])
        rb = [b for b in reliability_bins(p, y) if b['n'] >= 10]
        axes[0].plot([b['mean_confidence'] for b in rb], [b['accuracy'] for b in rb], ls, marker=mk, ms=3, label=f'{crop} {setname}')
        c, rk = risk_coverage(p, y); axes[1].plot(c, rk, ls, label=f'{crop} {setname}')
        acc = p.max(1) >= d['threshold']; axes[1].plot([acc.mean()], [1 - (p[acc].argmax(1) == y[acc]).mean() if acc.any() else np.nan], mk, color='k', ms=5)
axes[0].plot([0, 1], [0, 1], ':', color='grey'); axes[0].set_xlabel('Mean confidence (bins with at least 10 images)'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Reliability, deployed models (temperature-scaled)'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Coverage'); axes[1].set_ylabel('Selective risk'); axes[1].set_title('Risk and coverage (black marker: frozen threshold)'); axes[1].legend(fontsize=8)
for a in axes: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/figure_5_2_reliability_risk_coverage.png', dpi=200); plt.close()

def sheet(crop, setname, per=4, size=150):
    d = DEP[crop]; cl = CLASSES[crop]; df = SETS[(crop, setname)].copy(); y = np.array([cl.index(l) for l in df.research_label])
    p = softmax(np.load(f'{LOG}/{crop}_s{d["seed"]}_{d["format"]}_{setname}.npy'), d['temperature_T'])
    df['pred'] = [cl[i] for i in p.argmax(1)]; df['conf'] = p.max(1)
    df['outcome'] = np.where(df.conf < d['threshold'], 'rejected', np.where(df.pred == df.research_label, 'correct', 'wrong'))
    pick = df.groupby(['research_label', 'source_id', 'outcome'], group_keys=False).apply(lambda g: g.sample(min(per, len(g)), random_state=2026))
    pick[['sample_id', 'source_id', 'research_label', 'pred', 'conf', 'outcome']].to_csv(f'{OUT}/inspection_{crop}_{setname}.csv', index=False)
    cols = 8; rws = (len(pick) + cols - 1) // cols; sh = Image.new('RGB', (cols * size, 30 + rws * (size + 28)), 'white'); dr = ImageDraw.Draw(sh)
    dr.text((8, 8), f'{crop} {setname}: stratified sample of correct, wrong and rejected predictions (deployed model)', fill='black')
    for k, r in enumerate(pick.itertuples()):
        im = Image.open(f'{WORK}/{r.working_path}').convert('RGB'); im.thumbnail((size, size))
        x, yy = (k % cols) * size, 30 + (k // cols) * (size + 28); sh.paste(im, (x, yy))
        dr.text((x + 2, yy + size + 2), f'{r.outcome} {r.source_id} {r.research_label}>{r.pred} {r.conf:.2f}', fill='black')
    p_out = f'{OUT}/inspection_{crop}_{setname}.png'; sh.save(p_out); return p_out
SHEETS = [sheet(c, s) for c in CLASSES for s in ['internal', 'external']]
print('figures and sheets saved')

In [ ]:
# 9. Results file
res = '/content/nb05_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{OUT}/environment_nb05.json', f'{OUT}/results_all_variants.csv', f'{OUT}/results_seed_mean_sd.csv', f'{OUT}/results_detail.json',
              f'{OUT}/bootstrap_seed42.csv', f'{OUT}/maize_source_analysis.json', f'{OUT}/figure_5_1_confusion.png',
              f'{OUT}/figure_5_2_reliability_risk_coverage.png'] + SHEETS + glob.glob(f'{OUT}/inspection_*.csv'):
        if os.path.exists(p): z.write(p, os.path.basename(p))
from google.colab import files
files.download(res)

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb05_results.zip`. The copies used for the published results are in this repository under `records/nb05/`.